## Problem 1.1: Multi-Source Data Integration & Deduplication

**Context:** You have customer data from multiple sources (Oracle, Salesforce, API) with potential duplicates. You need to combine and deduplicate based on customer ID and email.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Create sample data from multiple sources
oracle_data = spark.createDataFrame([
    ("CUST001", "john.doe@example.com", "John Doe", "2024-01-15"),
    ("CUST002", "jane.smith@example.com", "Jane Smith", "2024-01-10"),
    ("CUST003", "bob.wilson@example.com", "Bob Wilson", "2024-01-12"),
], ["customer_id", "email", "name", "last_updated"])

salesforce_data = spark.createDataFrame([
    ("CUST001", "john.doe@example.com", "John Doe", "2024-01-20"),  # Duplicate
    ("CUST004", "alice.brown@example.com", "Alice Brown", "2024-01-18"),
    ("CUST002", "jane.smith@example.com", "Jane Smith", "2024-01-15"),  # Duplicate
], ["customer_id", "email", "name", "last_updated"])

api_data = spark.createDataFrame([
    ("CUST005", "charlie.davis@example.com", "Charlie Davis", "2024-01-19"),
    ("CUST001", "john.doe@example.com", "John Doe", "2024-01-18"),  # Duplicate
], ["customer_id", "email", "name", "last_updated"])

In [0]:
all_customers = oracle_data.union(salesforce_data).union(api_data)
display(all_customers)


In [0]:
window_spec = Window.partitionBy("customer_id","email").orderBy(col("last_updated").desc())


In [0]:
dbutils.help()

In [0]:
dbutils.widgets.help()

In [0]:
%sql
-- Step 1: Create the 'employee' table
DROP table IF EXISTS employee;
CREATE TABLE employee (
    emp_id INT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    department VARCHAR(50),
    salary DECIMAL(10, 2),
    hire_date DATE
);

-- Step 2: Insert data into the 'employee' table
INSERT INTO employee (emp_id, first_name, last_name, department, salary, hire_date) VALUES
(1, 'Steven', 'King', 'IT', 24000.00, '2003-06-17'),
(2, 'Neena', 'Kochhar', 'IT', 17000.00, '2005-09-21'),
(3, 'Lex', 'De Haan', 'IT', 17000.00, '2001-01-13'),
(4, 'Alexander', 'Hunold', 'Sales', 9000.00, '2006-01-03'),
(5, 'Bruce', 'Ernst', 'Sales', 6000.00, '2007-05-21'),
(6, 'David', 'Austin', 'Sales', 4800.00, '2005-06-25'),
(7, 'Valli', 'Pataballa', 'Sales', 4800.00, '2006-02-05'),
(8, 'Diana', 'Lorentz', 'Sales', 4200.00, '2007-02-07'),
(9, 'Nancy', 'Greenberg', 'HR', 12008.00, '2002-08-17'),
(10, 'Daniel', 'Faviet', 'HR', 9000.00, '2002-08-16');


In [0]:
%sql
SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS denserank,
              rank() OVER (ORDER BY salary DESC) AS rank,
              ROW_NUMBER() OVER (ORDER BY salary DESC) AS rownumber,
              LAG(salary) OVER (ORDER BY salary DESC) AS lag,
              LEAD(salary) OVER (ORDER BY salary DESC) AS lead
FROM employee;

In [0]:
%sql
with RankedSalaries AS (
   SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS denserank,
              rank() OVER (ORDER BY salary DESC) AS rank,
              ROW_NUMBER() OVER (ORDER BY salary DESC) AS rownumber,
              LAG(salary) OVER (ORDER BY salary DESC) AS lag,
              LEAD(salary) OVER (ORDER BY salary DESC) AS lead
FROM employee)
SELECT salary AS SecondHighestSalary
FROM RankedSalaries
WHERE denserank = 2;

-- WITH RankedSalaries AS (
-- SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rank
-- FROM employee
-- )
-- SELECT salary AS SecondHighestSalary
-- FROM RankedSalaries
-- WHERE rank = 2;

In [0]:
%sql
with rankedsalaries as ( 
        select salary, dense_rank() over (order by salary desc) as rank from employee
        ) 
select salary as secondhighestsalary from rankedsalaries where rank =2


In [0]:
%sql
with rankedsalaries as (
  select salary, dense_rank() over( order by salary desc) as rank from employee
)
select salary as secondhighestsalary from rankedsalaries where rank = 2;

Question 2: Write a SQL query to find the numbers which consecutively occur 3

In [0]:
%sql
DROP TABLE IF EXISTS table_name;
CREATE TABLE IF not EXISTS table_name (
    id INT,
    numbers INT
);


INSERT INTO table_name (id, numbers) VALUES
(1, 1),
(2, 1),
(3, 1),
(4, 2),
(5, 1),
(6, 2),
(7, 2),
(8, 2),
(9, 3),
(10, 3),
(11, 3),
(12, 3);

In [0]:
%sql
SELECT numbers,
LEAD(numbers, 1) OVER (ORDER BY id) AS next_num,
LEAD(numbers, 2) OVER (ORDER BY id) AS next_next_num,
LEAD(numbers, 3) OVER (ORDER BY id) AS next_next_num1
FROM table_name;

In [0]:
%sql
SELECT numbers
FROM (
SELECT numbers,
LEAD(numbers, 1) OVER (ORDER BY id) AS next_num,
LEAD(numbers, 2) OVER (ORDER BY id) AS next_next_num,
LEAD(numbers, 3) OVER (ORDER BY id) AS next_next_num1
FROM table_name
) t
WHERE numbers = next_num  
  AND numbers = next_next_num
  AND numbers = next_next_num1;

Question 3: Write a SQL query to find the days when temperature was higher than its previous dates. ● Table: table_name ● Columns: Days, Temp

In [0]:
%sql
drop table if exists table_name;
CREATE TABLE table_name (
    Days DATE,
    Temp INT
);
INSERT INTO table_name (Days, Temp) VALUES
('2024-01-01', 30),
('2024-01-02', 32),
('2024-01-03', 31),
('2024-01-04', 35),
('2024-01-05', 35),
('2024-01-06', 36);

In [0]:
%sql
    SELECT Days, Temp,
        LAG(Temp) OVER (ORDER BY Days) AS prev_temp
    FROM table_name;

In [0]:
%sql
SELECT Days, Temp
FROM (
    SELECT Days, Temp,
        LAG(Temp) OVER (ORDER BY Days) AS prev_temp
    FROM table_name
) t
WHERE Temp > prev_temp;

-- with TemnpWithPrev as (
--   select days,temp,
--         LAG(temp) over (order by days) As prev_temp
--   from table_name
-- )
-- select days,temp from TemnpWithPrev where temp > prev_temp;

: Write a SQL query to delete duplicate rows in a table.
● Table: table_name
● Columns: column1, column2, ..., columnN

In [0]:
%sql
DROP TABLE IF EXISTS table_name;
CREATE TABLE table_name (
    id INT,
    column1 VARCHAR(50),
    column2 VARCHAR(50)
);

INSERT INTO table_name (id, column1, column2) VALUES
(1, 'A', 'X'),
(2, 'A', 'X'),   -- duplicate of (A, X)
(3, 'A', 'X'),   -- duplicate of (A, X)
(4, 'B', 'Y'),
(5, 'B', 'Y'),   -- duplicate of (B, Y)
(6, 'C', 'Z');

In [0]:
%sql
SELECT MIN(id)
    FROM table_name
    GROUP BY column1, column2;

In [0]:
%sql
delete from table_name 
where id not in 
    (select min(id) from table_name group by column1, column2);

Write a SQL query for the cumulative sum of salary of each employee from January to July. ● Table: table_name ● Columns: Emp_id, Month, Salary

In [0]:
%sql
drop table if exists table_name;
CREATE TABLE table_name (
    Emp_id INT,
    Month VARCHAR(20),
    Salary INT
);

INSERT INTO table_name (Emp_id, Month, Salary) VALUES
(1, '2024-01-01', 3000),
(1, '2024-02-01', 3200),
(1, '2024-03-01', 3100),
(1, '2024-04-01', 3300),
(1, '2024-05-01', 3400),
(1, '2024-06-01', 3500),
(1, '2024-07-01', 3600),

(2, '2024-01-01', 4000),
(2, '2024-02-01', 4200),
(2, '2024-03-01', 4100);


In [0]:
%sql
SELECT Emp_id, Month, salary,
    SUM(Salary) OVER ( PARTITION BY Emp_id ORDER BY Month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) AS CumulativeSalary
FROM table_name;


Write a SQL query to display year-on-year growth for each product.
● Table: table_name
● Columns: transaction_id, Product_id, transaction_date, spend

In [0]:
%sql
drop Table if exists table_name;
CREATE TABLE table_name (
    transaction_id INT,
    product_id INT,
    transaction_date DATE,
    spend DECIMAL(10,2)
);

INSERT INTO table_name VALUES
(1, 101, '2022-03-10', 1000),
(2, 101, '2022-08-15', 1500),
(3, 101, '2023-02-20', 2000),
(4, 101, '2023-09-05', 2500),
(5, 101, '2024-04-12', 3000),

(6, 102, '2022-05-01', 800),
(7, 102, '2023-06-10', 1200),
(8, 102, '2024-07-22', 1800);

In [0]:
%sql
select * from table_name;

In [0]:
%sql
WITH yearly_spend AS (
    SELECT
        product_id,
        EXTRACT(YEAR FROM transaction_date) AS year,
        SUM(spend) AS total_spend
    FROM table_name
    GROUP BY product_id, EXTRACT(YEAR FROM transaction_date)
) FROM yearly_spend

In [0]:
%sql
WITH yearly_spend AS (
    SELECT
        product_id,
        EXTRACT(YEAR FROM transaction_date) AS year,
        SUM(spend) AS total_spend
    FROM table_name
    GROUP BY product_id, EXTRACT(YEAR FROM transaction_date)
),
yoy_calc AS (
    SELECT
        product_id,
        year,
        total_spend,
        LAG(total_spend) OVER (
            PARTITION BY product_id
            ORDER BY year
        ) AS prev_year_spend
    FROM yearly_spend
)
SELECT
    product_id,
    year,
    total_spend,
    prev_year_spend,
    ROUND(
        (total_spend - prev_year_spend) * 100.0 / prev_year_spend,
        2
    ) AS yoy_growth_percent
FROM yoy_calc
WHERE prev_year_spend IS NOT NULL
ORDER BY product_id, year;

Question 8: Write a SQL query to get the emp_id and department for each
department where the most recently joined employee is still working.
● Table: table_name
● Columns: emp_id, first_name, last_name, date_of_join, date_of_exit,
department

In [0]:
%sql
Drop table if exists table_name;
CREATE TABLE table_name (
    emp_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    date_of_join DATE,
    date_of_exit DATE,
    department VARCHAR(50)
);

INSERT INTO table_name VALUES
(1, 'John',  'Doe',   '2021-01-10', NULL,         'HR'),
(2, 'Jane',  'Smith', '2022-03-15', '2023-06-01', 'HR'),
(3, 'Alice', 'Brown', '2023-07-01', NULL,         'HR'),

(4, 'Bob',   'White', '2020-02-20', NULL,         'IT'),
(5, 'Carol', 'Green', '2024-01-05', NULL,         'IT'),

(6, 'Dave',  'Black', '2023-05-10', '2024-02-01', 'Finance'),
(7, 'Eva',   'Gray',  '2022-08-12', NULL,         'Finance');

In [0]:
%sql
SELECT emp_id, department
FROM table_name
WHERE date_of_exit IS NULL
ORDER BY date_of_join DESC;

In [0]:
%sql
WITH ranked_emps AS (
    SELECT
        emp_id,
        department,
        ROW_NUMBER() OVER (
            PARTITION BY department
            ORDER BY date_of_join DESC
        ) AS rn
    FROM table_name
    WHERE date_of_exit IS NULL
)
SELECT emp_id, department
FROM ranked_emps
WHERE rn = 1;

Question 9: How many rows will come in the outputs of Left, Right, Inner, and
Outer Join from two tables having duplicate rows?

Top‑3 customers by spend per region in last 30 days Tables: orders(order_id, customer_id, region, order_date, amount).****

In [0]:
%sql
drop table if exists orders;

In [0]:
%sql

CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    region VARCHAR(50),
    order_date DATE,
    amount DECIMAL(10,2)
);

INSERT INTO orders VALUES
(1, 101, 'APAC', '2024-12-20', 500),
(2, 101, 'APAC', '2025-01-10', 700),
(3, 102, 'APAC', '2025-01-15', 1200),
(4, 103, 'APAC', '2025-01-18', 900),
(5, 104, 'APAC', '2025-01-20', 300),

(6, 201, 'EMEA', '2025-01-05', 1500),
(7, 202, 'EMEA', '2025-01-12', 800),
(8, 203, 'EMEA', '2025-01-22', 1100),
(9, 204, 'EMEA', '2025-01-25', 400),

(10, 301, 'US', '2025-01-08', 2000),
(11, 302, 'US', '2025-01-14', 1800),
(12, 303, 'US', '2025-01-18', 900),
(13, 304, 'US', '2025-01-19', 700);

In [0]:
%sql
with last_30_days AS (
    SELECT
        customer_id,
        region,
        SUM(amount) AS total_spend
    FROM orders
    WHERE order_date >= "2025-01-15" - INTERVAL '30' DAY
    GROUP BY customer_id, region
),
ranked_customers AS (
    SELECT
        customer_id,
        region,
        total_spend,
        DENSE_RANK() OVER (
            PARTITION BY region
            ORDER BY total_spend DESC
        ) AS rnk
    FROM last_30_days
)
SELECT
    region,
    customer_id,
    total_spend
FROM ranked_customers
WHERE rnk <= 3
ORDER BY region, total_spend DESC;

Question 10: Write a query to get mean, median, and mode for earnings.
● Table: table_name
● Columns: Emp_id, salary


In [0]:
%sql
DROP TABLE IF EXISTS table_name;
CREATE TABLE table_name (
    emp_id INT,
    salary INT
);

INSERT INTO table_name (emp_id, salary) VALUES
(1, 5000),
(2, 6000),
(3, 6000),
(4, 7000),
(5, 8000),
(6, 6000),
(7, 9000);

In [0]:
%sql
SELECT AVG(salary) AS MeanSalary FROM table_name;

In [0]:
%sql
WITH ordered_salaries AS (
    SELECT
        salary,
        ROW_NUMBER() OVER (ORDER BY salary) AS rn,
        COUNT(*) OVER () AS total_rows
    FROM table_name
)
SELECT
    AVG(salary) AS median_salary
FROM ordered_salaries
WHERE rn IN (
    (total_rows + 1) / 2,
    (total_rows + 2) / 2
);

In [0]:
%sql
SELECT salary AS ModeSalary
FROM table_name
GROUP BY salary
ORDER BY COUNT(*) DESC
LIMIT 1;

Write a SQL query to find the longest streak of consecutive days an
employee worked.
● Table: attendance
● Columns: emp_id, work_date

In [0]:
%sql
drop table if exists sales;
CREATE TABLE sales (
    product_category VARCHAR(50),
    sale_year INT,
    revenue DECIMAL(10,2)
);

INSERT INTO sales VALUES
('Electronics', 2024, 50000),
('Electronics', 2024, 30000),
('Clothing',    2024, 20000),
('Clothing',    2024, 15000),
('Furniture',   2024, 25000),
('Furniture',   2024, 10000),

('Electronics', 2023, 40000),
('Clothing',    2023, 18000),
('Furniture',   2023, 22000);

In [0]:
%sql
WITH category_sales AS (
    SELECT
        product_category,
        SUM(revenue) AS category_revenue
    FROM sales
    WHERE sale_year = 2024
    GROUP BY product_category
)

SELECT
    product_category,
    category_revenue,
    ROUND(
        category_revenue * 100.0 / SUM(category_revenue) OVER (),
        2
    ) AS percentage_of_total_sales
FROM category_sales;

Write a SQL query to calculate the percentage of total sales contributed by each product category in a given year. ● Table: sales ● Columns: product_category, sale_year, revenue

In [0]:
%sql
drop table if exists sales;
CREATE TABLE sales (
    product_category VARCHAR(50),
    sale_year INT,
    revenue DECIMAL(10,2)
);

INSERT INTO sales VALUES
('Electronics', 2024, 50000),
('Electronics', 2024, 30000),
('Clothing',    2024, 20000),
('Clothing',    2024, 15000),
('Furniture',   2024, 25000),
('Furniture',   2024, 10000),

('Electronics', 2023, 40000),
('Clothing',    2023, 18000),
('Furniture',   2023, 22000);

In [0]:
%sql
WITH category_sales AS (
    SELECT
        product_category,
        SUM(revenue) AS category_revenue
    FROM sales
    WHERE sale_year = 2024
    GROUP BY product_category
)
SELECT
    product_category,
    category_revenue,
    ROUND(
        category_revenue * 100.0 / SUM(category_revenue) OVER (),
        2
    ) AS percentage_of_total_sales
FROM category_sales;

Write a SQL query to find the longest streak of consecutive days an
employee worked.
● Table: attendance
● Columns: emp_id, work_date

In [0]:
%sql

drop table if exists attendance;
CREATE TABLE attendance (
    emp_id INT,
    work_date DATE
);

INSERT INTO attendance VALUES
-- Employee 1
(1, '2024-01-01'),
(1, '2024-01-02'),
(1, '2024-01-03'),
(1, '2024-01-05'),
(1, '2024-01-06'),
(1, '2024-01-07'),
(1, '2024-01-10'),

-- Employee 2
(2, '2024-01-01'),
(2, '2024-01-03'),
(2, '2024-01-04'),
(2, '2024-01-05');

In [0]:
%sql
WITH ConsecutiveDays AS (
SELECT emp_id, work_date,
ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY work_date) -
DENSE_RANK() OVER (PARTITION BY emp_id, DATE_ADD(work_date, -ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY work_date)) ORDER BY work_date) AS streak_group
FROM attendance
)
SELECT emp_id, COUNT(*) AS longest_streak
FROM ConsecutiveDays
GROUP BY emp_id, streak_group
ORDER BY longest_streak DESC
LIMIT 1;


In [0]:
%sql
WITH numbered_days AS (
    SELECT
        emp_id,
        work_date,
        ROW_NUMBER() OVER ( PARTITION BY emp_id ORDER BY work_date ) AS rn
    FROM attendance
),
streaks AS (
    SELECT
        emp_id,
        work_date,
        DATEADD(day, -rn, work_date) AS streak_group
    FROM numbered_days
)
SELECT
    emp_id,
    COUNT(*) AS longest_streak
FROM streaks
GROUP BY emp_id, streak_group
ORDER BY emp_id, longest_streak DESC;

In [0]:
%sql
WITH ConsecutiveDays AS (
SELECT emp_id, work_date,
ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY work_date) -
DENSE_RANK() OVER (PARTITION BY emp_id, DATE_ADD(work_date, -ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY work_date)) ORDER BY work_date) AS streak_group
FROM attendance
)
SELECT emp_id, COUNT(*) AS longest_streak
FROM ConsecutiveDays
GROUP BY emp_id, streak_group
ORDER BY longest_streak DESC
LIMIT 1;


In [0]:
%sql
drop table if exists A;
CREATE TABLE A (val INT);

INSERT INTO A VALUES
(0),
(0),
(1),
(NULL);

drop table if exists B;

CREATE TABLE B (val INT);

INSERT INTO B VALUES
(0),
(1),
(1),
(NULL);

In [0]:
%sql
SELECT *
FROM A
INNER JOIN B
ON A.val = B.val;

In [0]:
%sql
SELECT *
FROM A
LEFT JOIN B
ON A.val = B.val;

In [0]:
%sql
SELECT *
FROM A
RIGHT JOIN B
ON A.val = B.val;

In [0]:
%sql
SELECT *
FROM A
FULL OUTER JOIN B
ON A.val = B.val;